In [1]:
from datasets import load_dataset

ds = load_dataset("lmms-lab/POPE", "default")

/opt/conda/envs/lmms-eval/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds

DatasetDict({
    test: Dataset({
        features: ['id', 'question_id', 'question', 'answer', 'image_source', 'image', 'category'],
        num_rows: 9000
    })
})

In [3]:
ds['test'][1]

{'id': '1',
 'question_id': '2',
 'question': 'Is there a backpack in the image?',
 'answer': 'no',
 'image_source': 'COCO_val2014_000000310196',
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x427>,
 'category': 'adversarial'}

In [ ]:
# show the entire dataset
import pandas as pd

df = pd.DataFrame(ds)
df.head()

### Push custom datasets to HF

In [2]:
import json

org_ann_file_path = '/root/Desktop/workspace/miso/hub/datasets/lavis/rlhf_v_augmented/annotations/final_target.json'

# create new annotation file
new_ann_list = []

with open(org_ann_file_path, 'r') as f:
    org_ann_list = json.load(f)

for idx, ann in enumerate(org_ann_list):
    new_ann_list.append({
        'idx': idx,
		'image_path': ann['image_path'],
        'text_input': ann['text_input'],
        'negative_target_word': ann['negative_target_word'],
		'positive_target_word': ann['positive_target_word'],
    })

new_ann_list[:2]

[{'idx': 0,
  'image_path': 'llava1.5_raw_images/00013/000139279.jpg',
  'text_input': 'What are the key features you observe in the image?  A young man standing on stage wearing white {target word}',
  'negative_target_word': 'pants',
  'positive_target_word': 'shirt'},
 {'idx': 1,
  'image_path': 'coco2017/train2017/000000387153.jpg',
  'text_input': 'Are the countertops a light or dark color? The countertop in the picture is {target word}',
  'negative_target_word': 'white',
  'positive_target_word': 'dark'}]

In [3]:
# save new annotation file
import json

ann_file_path = '/root/Desktop/workspace/miso/hub/datasets/lavis/rlhf_v_augmented/annotations/annotations.json'

with open(ann_file_path, 'w') as f:
    json.dump(new_ann_list, f, indent=4)

In [ ]:
import pandas as pd
from datasets import Dataset, Features, Value, Array2D, ClassLabel, Image

# push new annotation file to HF

# convert to DataFrame format
rows = []
for ann in new_ann_list:
	rows.append({
		'idx': idx,
		'image_path': ann['image_path'],
        'text_input': ann['text_input'],
        'negative_target_word': ann['negative_target_word'],
		'positive_target_word': ann['positive_target_word'],
	})
df = pd.DataFrame(rows)

# features
features = Features({
	"idx": Value("string"),
	"image": Image(),
	"text_input": Value("string"),
	"negative_target_word": Value("string"),
	"positive_target_word": Value("string"),
})

hf_ds = Dataset.from_pandas(df) # , features=features

In [7]:
hf_ds

Dataset({
    features: ['idx', 'image_path', 'text_input', 'negative_target_word', 'positive_target_word'],
    num_rows: 292
})

In [8]:
from huggingface_hub import HfApi
api = HfApi()

api.upload_file(
  path_or_fileobj=ann_file_path,
  path_in_repo="annotations.json",
  repo_id="miso-choi/RLHF-V-Aug",
  repo_type="dataset"
)

CommitInfo(commit_url='https://huggingface.co/datasets/miso-choi/RLHF-V-Aug/commit/96c0096814fc28fe3473aec0315aafa9e9e7ff67', commit_message='Upload annotations.json with huggingface_hub', commit_description='', oid='96c0096814fc28fe3473aec0315aafa9e9e7ff67', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/miso-choi/RLHF-V-Aug', endpoint='https://huggingface.co', repo_type='dataset', repo_id='miso-choi/RLHF-V-Aug'), pr_revision=None, pr_num=None)

In [ ]:
image_folder_path = '/root/Desktop/workspace/miso/hub/datasets/lavis/rlhf_v_augmented/images'
api.upload_folder(
  folder_path=image_folder_path,
  path_in_repo="images",
  repo_id="miso-choi/RLHF-V-Aug",
  repo_type="dataset"
)

In [37]:
from datasets import load_dataset
ds = load_dataset(
	"miso-choi/RLHF-V-Aug",
	data_dir = "images",
	split=None
	)
ds


DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 3716
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 100
    })
})

In [24]:
ds.keys()

dict_keys(['train', 'validation'])

In [25]:
len(ds['train'])

3716

In [29]:
from datasets import load_dataset
ann = load_dataset(
	"miso-choi/RLHF-V-Aug",
	data_files = "annotations.json"
	)
ann


DatasetDict({
    train: Dataset({
        features: ['idx', 'image_path', 'text_input', 'negative_target_word', 'positive_target_word'],
        num_rows: 292
    })
})

In [33]:
ann

datasets.dataset_dict.DatasetDict

In [34]:
import pandas as pd

df = pd.DataFrame(ann)
df.head()

,train
0,"{'idx': 0, 'image_path': 'llava1.5_raw_images/..."
1,"{'idx': 1, 'image_path': 'coco2017/train2017/0..."
2,"{'idx': 2, 'image_path': 'coco2017/train2017/0..."
3,"{'idx': 3, 'image_path': 'coco2017/train2017/0..."
4,"{'idx': 4, 'image_path': 'coco2017/train2017/0..."


In [36]:
ann['train'][0]

{'idx': 0,
 'image_path': 'llava1.5_raw_images/00013/000139279.jpg',
 'text_input': 'What are the key features you observe in the image?  A young man standing on stage wearing white {target word}',
 'negative_target_word': 'pants',
 'positive_target_word': 'shirt'}

In [5]:
from datasets import load_dataset
ds = load_dataset("miso-choi/RLHF-V-Aug", split="all", download_mode="force_redownload")

ds



RuntimeError: Dataset scripts are no longer supported, but found RLHF-V-Aug.py

In [11]:
len(ds)

3816